In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
PGVECTOR_ID = os.getenv("PGVECTOR_ID")
PGVECTOR_PW = os.getenv("PGVECTOR_PW")
PGVECTOR_HOST = os.getenv("PGVECTOR_HOST")
PGVECTOR_PORT = os.getenv("PGVECTOR_PORT")
PGVECTOR_DB = os.getenv("PGVECTOR_DB")

In [6]:
connection = f"postgresql+psycopg://{PGVECTOR_ID}:{PGVECTOR_PW}@{PGVECTOR_HOST}:{PGVECTOR_PORT}/{PGVECTOR_DB}"

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
docs = loader.load()

C:\Users\user\AppData\Local\Temp\ipykernel_22240\2084735666.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

In [10]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [13]:
from langchain_postgres import PGVector

db = PGVector.from_documents(
    texts, embeddings, connection=connection
)

In [15]:
query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

In [16]:
results = db.similarity_search(query, k=3)

In [17]:
results

[Document(id='a2a4ef68-34fb-428a-afc0-690de378504d', metadata={'page': 16, 'title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx', 'author': '', 'format': 'PDF 1.7', 'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'creator': '', 'modDate': "D:20260116002528+09'00'", 'moddate': '2026-01-16T00:25:28+09:00', 'subject': '', 'trapped': '', 'keywords': '', 'producer': 'Microsoft: Print To PDF', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'total_pages': 22, 'creationDate': "D:20260116002528+09'00'", 'creationdate': '2026-01-16T00:25:28+09:00'}, page_content='7.2 자격증 취득 지원 \n직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다. \n자격 등급 \n축하금 (1 회성)\n자격 수당 (월)\n대상 자격증 예시 \n기술사/기능장\n200 만원 \n30 만원 \n금속재료, 용접, 기계가공 등\n기사 \n50 만원 \n10 만원 \n일반기계, 전기, 산업안전 등\n산업기사 \n30 만원 \n5 만원 \n기계설계, 위험물 등 \n기능사 \n10 만원 \n3 만원 \n선반, 밀링, 특수용접 등 \n\uf0b7 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수 \n제한 없음. \n7.3 해외 연수 (Global Explorer) \n\uf0b7 대상: 연간 최우수 사원 (MVP) 및 우수 팀. \n\uf0b7 내용: 매년 1

In [18]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [19]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [20]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0,
)

In [21]:
chain = prompt | llm | parser

In [22]:
response = chain.invoke({"context": results, "question": query})

In [23]:
response

'컨텍스트에 따르면, 국가 기술 자격 중 기사 자격증을 취득하면 다음과 같은 혜택을 받을 수 있습니다.\n\n*   **축하금 (1 회성):** 50 만원\n*   **자격 수당 (월):** 10 만원\n\n따라서, 국가 기술 자격 중 기사 자격증을 취득하면 **월 10 만원 자격 수당과 1 회성 축하금 50 만원**을 받을 수 있습니다.'